In [41]:
%load_ext autoreload
%autoreload 2

import numpy as np
import sys
np.set_printoptions(threshold=sys.maxsize)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
from src.config import Configuration
from src.tetris import TetrisConfiguration

T_CONFIG = TetrisConfiguration(
    board_w=4,
    board_h=10,
    vanish_zone=4, # Extra rows above the visible board to capture piece spawns
)

CONFIG = Configuration(
    max_placements=50,
    max_board_size_w=30,
    max_board_size_h=10,
)

# Train

In [43]:
from src.tetris import Board, PieceEnum, Queue, ActionEnum, ActivePiece, Tetris, RotationEnum, ROTATION_DIR

In [44]:
from src.models import TetrisEnv

env = TetrisEnv(CONFIG, T_CONFIG)

In [45]:
print(env.reset()[0]["boards"].shape)
print(env.reset()[0]["queue"].shape)

(50, 14, 30)
(7, 8)


### Model

In [46]:
env.observation_space["boards"].shape

(50, 14, 30)

In [47]:
from src.models import TetrisFeatureExtractor

feature_extractor = TetrisFeatureExtractor(
    T_CONFIG,
    CONFIG,
    env.observation_space,
)

### Correct forward pass (current model)

The feature extractor returns a single tensor `(B, max_placements)` — one scalar per placement.
The env provides a `placement_mask` so the model masks out invalid/padded slots with `-1e9`.

In [48]:
import torch

obs, _ = env.reset()
tensor_obs = {
    "boards": torch.as_tensor(obs["boards"], dtype=torch.float32).unsqueeze(0),
    "queue":  torch.as_tensor(obs["queue"], dtype=torch.float32).unsqueeze(0),
    "placement_mask": torch.as_tensor(obs["placement_mask"], dtype=torch.bool).unsqueeze(0),
}

values = feature_extractor(tensor_obs)
print("Output shape:", values.shape)                # (1, 50)
print("Mask:", tensor_obs["placement_mask"][0])
print("Valid placements:", tensor_obs["placement_mask"].sum().item())
print("Placement values:\n", values)
print("Best placement:", values.argmax().item())

Output shape: torch.Size([1, 50])
Mask: tensor([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False])
Valid placements: 10
Placement values:
 tensor([[ 2.3501e-02,  2.3665e-02,  2.3444e-02,  2.3525e-02,  2.3347e-02,
          2.3412e-02,  2.3247e-02,  2.3523e-02,  2.3266e-02,  2.3399e-02,
         -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09,
         -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09,
         -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09,
         -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09,
         -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09,
         -1